# Lab 4 — Context Management

Configure `context_manager="auto"`; explore large synthetic tool output and observe whether your installed SDK triggers reduction. Compression is conditional and is **not guaranteed**.

**Run cells from top to bottom.** Execute `00_Setup.ipynb` first. Each notebook is intended to be opened from the `notebooks/` folder.

In [1]:
import os, sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from strands import Agent, tool
from common import INCIDENT

## 1. Create a read-only large-output tool

Begin small and increase only as needed. Very large requests may incur latency and model costs.

In [2]:
@tool
def get_synthetic_large_logs(repeat_count: int = 100) -> str:
    """Return repeated synthetic incident log lines for testing context growth; no AWS access."""
    repeat_count = min(max(1, repeat_count), 1200)
    lines = INCIDENT["log_samples"] * repeat_count
    return json.dumps({"incident_id": "INC-1042", "lines": lines})

print("Example characters:", len(get_synthetic_large_logs.__name__))
print("Tool ready")

Example characters: 24
Tool ready


## 2. Configure an Agent with automatic context management

In [3]:
context_agent = Agent(
    name="context_demo_agent",
    system_prompt="Analyze only synthetic incident evidence. Call get_synthetic_large_logs when asked; summarize evidence with uncertainty. Do not call AWS.",
    tools=[get_synthetic_large_logs],
    context_manager="auto",
)
print("Context manager configured")

Context manager configured


## 3. Execute the experiment

Choose a moderate repeat count for initial training. Higher repetition can raise model cost, time, and overflow risk.

In [4]:
REPEAT_COUNT = 100  # increase slowly after validating access and costs
response = context_agent(f"Call get_synthetic_large_logs with repeat_count={REPEAT_COUNT} and summarize recurring errors in INC-1042.")
print(response)

Sure! Let me fetch the synthetic logs right away.
Tool #1: get_synthetic_large_logs
The logs were large enough that part of the content was offloaded. Let me retrieve the elided section to ensure a complete analysis.
Tool #2: retrieve_context
I now have the full picture across both the initial fetch and the retrieved offloaded content. Here is a thorough summary:

---

## 🔍 INC-1042 — Recurring Error & Warning Summary

> **Source:** Synthetic logs, `repeat_count=100` | **Service:** `checkout-api`
> ⚠️ *These are synthetic incident logs. Findings are based solely on this evidence and carry inherent uncertainty about real-world applicability.*

---

### 📋 Event Breakdown (per 100-iteration cycle)

Each iteration produces exactly **3 repeating log lines**, cycling continuously across the entire dataset:

| # | Timestamp | Level | Message | Est. Occurrences |
|---|-----------|-------|---------|-----------------|
| 1 | `08:03:10` | **ERROR** | `checkout-api database connection acquisition t

## 4. Validate actual behavior

Successful output only proves that the Agent completed. To prove truncation, look for context-management logs/traces or specific events supported by your installed SDK. Compare with `context_manager=False` in a separate fresh Agent **only if** you are comfortable with extra token costs.

In [ ]:
print("Configured mode: auto")
print("Context reduction observed? Record logs/traces manually: YES / NO / NOT TRIGGERED")
print("Observation: ___________________________________________________")

**Checkpoint:** You can explain the difference between configuring auto mode and verifying that it actually compressed content. ContextManager is experimental; review SDK documentation before production use.